# English → Indic NMT with OpenNMT-py + pretrained embeddings (FastText vs BPEmb)
Same ILCI subset of `ai4bharat/BPCC` and the same length-binned splits as the previous lab. A Transformer is trained with **OpenNMT-py** twice, once per embedding type:

1. **FastText** (word-level): pretrained wiki vectors from <https://fasttext.cc/docs/en/pretrained-vectors.html>
2. **BPEmb** (subword-level): text is first encoded into subwords with BPEmb's SentencePiece model, and BPEmb's pretrained subword vectors initialise the embedding layers (<https://github.com/bheinzerling/bpemb>)

Steps: (1) load data, (2) length-binned train/valid/test splits (4 bins), (3) build the embeddings for each experiment, (4) train a Transformer, (5) translate the test set, (6) sacreBLEU overall and bin-wise, (7) compare the two embeddings and analyse.

## 0. Environment check
OpenNMT-py 3.x needs `torch<2.3` and `pyonmttok`, which only exist for **Python ≤ 3.12 / 3.11**. If this notebook's Python is newer (for example 3.13 on Kaggle), the cell below automatically creates a separate **Python 3.11** environment just for OpenNMT-py (using `uv`) and runs the OpenNMT commands there. Everything else (data, BPEmb, sacreBLEU) stays in the notebook's own Python.

In [1]:
import sys, subprocess, platform, os, glob, re, shutil, time
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"   # avoids a common OpenMP crash on macOS
print(sys.executable)
print("Machine:", platform.machine(), "| Python:", platform.python_version())

def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

def can_import(py, modules):
    r = subprocess.run([py, "-c", f"import {modules}"], capture_output=True, text=True)
    return r.returncode == 0

# --- packages used directly in this notebook (no pin on numpy: works on every Python version) ---
pip_install("sentencepiece", "sacrebleu", "datasets", "huggingface_hub", "pandas", "bpemb", "requests", "tqdm")

# --- OpenNMT-py: install in this Python if it is old enough, otherwise in a separate Python 3.11 venv ---
ONMT_VENV = os.path.abspath("onmt_py311")
if sys.version_info[:2] <= (3, 11):
    pip_install("numpy<2", "OpenNMT-py")
    ONMT_PY = sys.executable
else:
    ONMT_PY = os.path.join(ONMT_VENV, "Scripts" if os.name == "nt" else "bin", "python")
    if not (os.path.exists(ONMT_PY) and can_import(ONMT_PY, "onmt, pyonmttok")):
        print("Python", platform.python_version(), "is too new for OpenNMT-py -> creating a Python 3.11 environment (takes a few minutes)...")
        pip_install("uv")
        subprocess.run([sys.executable, "-m", "uv", "venv", "--python", "3.11", ONMT_VENV], check=True)
        subprocess.run([sys.executable, "-m", "uv", "pip", "install", "--python", ONMT_PY, "numpy<2", "OpenNMT-py"], check=True)
print("OpenNMT-py python:", ONMT_PY)

assert can_import(ONMT_PY, "onmt, pyonmttok"), "OpenNMT-py import failed in " + ONMT_PY
assert can_import(sys.executable, "bpemb, sacrebleu"), "bpemb / sacrebleu import failed"
r = subprocess.run([ONMT_PY, "-c", "import torch; print(torch.cuda.is_available())"], capture_output=True, text=True)
USE_GPU = r.stdout.strip() == "True"
print("Import checks: OK | CUDA available:", USE_GPU)

/usr/bin/python3
Machine: x86_64 | Python: 3.13.15
Python 3.13.15 is too new for OpenNMT-py -> creating a Python 3.11 environment (takes a few minutes)...


Using CPython 3.11.17
Creating virtual environment at: onmt_py311
Activate with: source onmt_py311/bin/activate
Using Python 3.11.17 environment at: onmt_py311
Resolved 91 packages in 943ms
Prepared 91 packages in 37.89s
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 91 packages in 6.02s
 + absl-py==2.5.1
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + anyio==4.16.0
 + blinker==1.9.0
 + blis==1.3.4
 + catalogue==2.0.10
 + certifi==2026.10.23
 + charset-normalizer==3.5.2
 + click==8.5.0
 + cloudpathlib==0.26.0
 + colorama==0.4.6
 + confection==1.3.4
 + configargparse==1.9.0
 + ctranslate2==4.8.5
 + cymem==2.0.16
 + fasttext-wheel==0.9.2
 + filelock==4.2.7
 + flask==3.1.3
 + fsspec==2026.9.0
 + grpcio==1.85.1
 + h11==0.16.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + idna==3.20
 + itsdangerous==2.2.0
 +

OpenNMT-py python: /kaggle/working/onmt_py311/bin/python
Import checks: OK | CUDA available: True


## 1. Configuration

In [2]:
SOURCE_LANG = "eng_Latn"
TARGET_LANG = "hin_Deva"      # same language as the previous lab (change to your native language)

# --- data (identical to the previous lab so the splits are the same) ---
TEST_FRAC   = 0.10
SEED        = 42
MAX_TRAIN_PER_BIN = 5000      # None = use all
MAX_TEST_PER_BIN  = 500       # None = use all
VALID_PER_BIN     = 100

# --- embeddings ---
EXPERIMENTS     = ["fasttext", "bpemb"]   # run both embedding types
EMB_DIM         = 300         # both FastText wiki vectors and BPEmb offer 300-d vectors
FT_VOCAB_LIMIT  = 200000      # keep only the 200k most frequent FastText vectors (file is sorted by frequency)
WORD_VOCAB_SIZE = 30000       # max vocabulary for the word-level (FastText) model
BPEMB_VS        = 10000       # BPEmb subword vocabulary size (1000/3000/5000/10000/25000/50000/100000/200000)
FREEZE_EMB      = False       # False = embeddings are fine-tuned, True = frozen

# --- Transformer / training ---
TRAIN_STEPS = 5000
LAYERS, HEADS, FF_SIZE = 3, 6, 1024     # hidden size = EMB_DIM, so HEADS must divide EMB_DIM
BEAM_SIZE   = 5

FORCE_RERUN = False           # False = skip training/translation if their outputs already exist
WORKDIR     = "mt_emb_work"

# ISO codes used by FastText wiki vectors and BPEmb
LANG2ISO = {"eng_Latn": "en", "hin_Deva": "hi", "ben_Beng": "bn", "guj_Gujr": "gu", "kan_Knda": "kn",
            "mal_Mlym": "ml", "mar_Deva": "mr", "pan_Guru": "pa", "tam_Taml": "ta", "tel_Telu": "te",
            "urd_Arab": "ur", "asm_Beng": "as", "ory_Orya": "or"}
ISO_SRC, ISO_TGT = LANG2ISO[SOURCE_LANG], LANG2ISO[TARGET_LANG]
assert EMB_DIM % HEADS == 0, "HEADS must divide EMB_DIM"

os.makedirs(WORKDIR, exist_ok=True)
EMB_DIR = os.path.join(WORKDIR, "embeddings")
os.makedirs(EMB_DIR, exist_ok=True)

## 2. Load the ILCI subset

In [3]:
from huggingface_hub import login, list_repo_files, hf_hub_download
import pandas as pd
import numpy as np
login()

files = list_repo_files("ai4bharat/BPCC", repo_type="dataset")
cands = [f for f in files if "ilci" in f.lower() and TARGET_LANG in f]
print("Matching files:", cands)
assert cands, "No ILCI file found for this language."

dfs = []
for f in cands:
    p = hf_hub_download("ai4bharat/BPCC", f, repo_type="dataset")
    if f.endswith(".parquet"):   d = pd.read_parquet(p)
    elif f.endswith(".csv"):     d = pd.read_csv(p)
    elif f.endswith(".tsv"):     d = pd.read_csv(p, sep="\t")
    else:                        d = pd.read_json(p, lines=True)
    dfs.append(d)
df = pd.concat(dfs, ignore_index=True)
print(df.columns.tolist(), len(df))

if "src_lang" in df.columns:
    fwd = df[(df.src_lang == SOURCE_LANG) & (df.tgt_lang == TARGET_LANG)]
    rev = df[(df.src_lang == TARGET_LANG) & (df.tgt_lang == SOURCE_LANG)].rename(columns={"src": "tgt", "tgt": "src"})
    df = pd.concat([fwd, rev], ignore_index=True)
df = df[["src", "tgt"]].dropna()
df["src"] = df["src"].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)
df["tgt"] = df["tgt"].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)
df = df[(df.src != "") & (df.tgt != "")].drop_duplicates().reset_index(drop=True)
print("Sentence pairs:", len(df))

Matching files: ['ilci/hin_Deva.tsv']


ilci/hin_Deva.tsv: reconstructing file:   0%|          |  0.00B / 62.6MB            

ilci/hin_Deva.tsv: downloading bytes:           |  0.00B            

['src_lang', 'tgt_lang', 'src', 'tgt'] 165649
Sentence pairs: 165649


## 3. Length bins and train / valid / test splits
Length = whitespace words in the English sentence. bin1: ≤10, bin2: 11–20, bin3: 21–29, bin4: ≥30. Each bin is split separately (same seed and procedure as the previous lab). A small **validation** set is carved from the non-test data, so the test set stays untouched.

In [4]:
def get_bin(n):
    if n <= 10: return "bin1"
    if n <= 20: return "bin2"
    if n < 30:  return "bin3"
    return "bin4"

df["len"] = df["src"].str.split().str.len()
df["bin"] = df["len"].apply(get_bin)

train_parts, valid_parts, test_parts = [], [], []
for b, g in df.groupby("bin"):
    g = g.sample(frac=1.0, random_state=SEED)
    n_test = max(1, int(round(len(g) * TEST_FRAC)))
    te, rest = g.iloc[:n_test], g.iloc[n_test:]
    va, tr = rest.iloc[:VALID_PER_BIN], rest.iloc[VALID_PER_BIN:]
    if MAX_TEST_PER_BIN is not None:  te = te.iloc[:MAX_TEST_PER_BIN]
    if MAX_TRAIN_PER_BIN is not None: tr = tr.iloc[:MAX_TRAIN_PER_BIN]
    test_parts.append(te); valid_parts.append(va); train_parts.append(tr)

train_df = pd.concat(train_parts).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
valid_df = pd.concat(valid_parts).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
test_df  = pd.concat(test_parts).reset_index(drop=True)

print(pd.DataFrame({"train": train_df["bin"].value_counts().sort_index(),
                    "valid": valid_df["bin"].value_counts().sort_index(),
                    "test":  test_df["bin"].value_counts().sort_index()}))

      train  valid  test
bin                     
bin1   5000    100   500
bin2   5000    100   500
bin3   5000    100   500
bin4   5000    100   500


## 4. Text helpers
* `word_tok` splits punctuation off words (used for the word-level FastText model and for evaluation). It deliberately does **not** use `\w`, which would cut Devanagari words at the vowel signs.
* `norm_eval` is applied to both the references and the hypotheses before BLEU, for **both** systems. It lower-cases, maps digits to `0` (BPEmb normalises text this way internally) and separates punctuation, so the Hindi danda `।` is compared fairly and the two embedding types are scored on exactly the same basis.

In [5]:
from sacrebleu.metrics import BLEU, CHRF
import requests

PUNCT_RE = re.compile(r"([.,;:!?()\[\]{}\"'\u201c\u201d\u2018\u2019`\u0964\u0965|/\\%\-\u2013\u2014\u2026])")

def word_tok(s):
    return PUNCT_RE.sub(r" \1 ", s).split()

def norm_eval(s):
    return " ".join(word_tok(re.sub(r"\d", "0", s.lower())))

def write_lines(path, lines):
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")

print(word_tok("Hearing loss, in 2020 (mild)."))
print(word_tok("यह एक परीक्षण वाक्य है।"))

['Hearing', 'loss', ',', 'in', '2020', '(', 'mild', ')', '.']
['यह', 'एक', 'परीक्षण', 'वाक्य', 'है', '।']


## 5. Embedding resources
**FastText:** the official wiki vectors (`wiki.<lang>.vec`, word2vec text format) are huge (several GB for English). The `.vec` files are sorted by word frequency, so I stream the file and keep only the first `FT_VOCAB_LIMIT` valid lines, which is all the model needs.

**BPEmb:** `BPEmb(lang=..., vs=..., dim=...)` provides the SentencePiece model (`.encode`) and the matching pretrained subword vectors. I write the vectors to a word2vec-format text file so OpenNMT-py can load them with the same mechanism (`src_embeddings` / `tgt_embeddings`).

In [6]:
FT_URL = "https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.{lang}.vec"

def fetch_fasttext_vec(lang, limit, out_path):
    """Stream wiki.<lang>.vec and keep the `limit` most frequent valid vectors (word2vec text format)."""
    if os.path.exists(out_path) and not FORCE_RERUN:
        print("FastText file exists:", out_path); return out_path
    url = FT_URL.format(lang=lang)
    body_path, kept = out_path + ".body", 0
    print("Streaming", url, "(stops after", limit, "vectors)")
    with requests.get(url, stream=True, timeout=120) as resp, open(body_path, "w", encoding="utf-8") as body:
        resp.raise_for_status()
        lines = resp.iter_lines(chunk_size=1 << 20)
        dim = int(next(lines).decode("utf-8").split()[1])
        for raw in lines:
            parts = raw.decode("utf-8", errors="ignore").rstrip().split(" ")
            if len(parts) != dim + 1:          # a few tokens contain spaces: skip them
                continue
            try:
                [float(x) for x in parts[1:]]
            except ValueError:
                continue
            body.write(" ".join(parts) + "\n")
            kept += 1
            if kept % 50000 == 0: print("  kept", kept)
            if kept >= limit: break
    with open(out_path, "w", encoding="utf-8") as out, open(body_path, encoding="utf-8") as body:
        out.write(f"{kept} {dim}\n")
        shutil.copyfileobj(body, out)
    os.remove(body_path)
    print(f"Saved {kept} vectors of dim {dim} -> {out_path}")
    return out_path

def write_w2v(path, tokens, vectors):
    """Write tokens + vectors in word2vec text format (header line, then 'token v1 ... vd')."""
    skip = {"<unk>", "<s>", "</s>", "<blank>", "<pad>"}
    keep = [i for i, t in enumerate(tokens) if t not in skip and len(t.split()) == 1 and " " not in t]
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"{len(keep)} {vectors.shape[1]}\n")
        for i in keep:
            f.write(tokens[i] + " " + " ".join(f"{x:.5f}" for x in vectors[i]) + "\n")
    print(f"Saved {len(keep)} vectors -> {path}")
    return path

def read_vec_tokens(path):
    toks = set()
    with open(path, encoding="utf-8") as f:
        next(f)
        for line in f:
            toks.add(line.split(" ", 1)[0])
    return toks

def load_bpemb(iso):
    from bpemb import BPEmb
    return BPEmb(lang=iso, vs=BPEMB_VS, dim=EMB_DIM, cache_dir=os.path.join(EMB_DIR, "bpemb_cache"))

## 6. Experiment set-up per embedding type
For each embedding type this returns the source/target tokenisers, the detokeniser and the embedding files.
* **fasttext**: lower-cased word tokens on both sides (word-level vocabulary).
* **bpemb**: raw text → BPEmb subwords (English model for the source, Hindi model for the target). Hypotheses are decoded back to text with the target SentencePiece model.

In [7]:
def get_setup(exp):
    if exp == "fasttext":
        src_vec = fetch_fasttext_vec(ISO_SRC, FT_VOCAB_LIMIT, os.path.join(EMB_DIR, f"ft.{ISO_SRC}.vec"))
        tgt_vec = fetch_fasttext_vec(ISO_TGT, FT_VOCAB_LIMIT, os.path.join(EMB_DIR, f"ft.{ISO_TGT}.vec"))
        src_tok = lambda s: word_tok(s.lower())
        tgt_tok = lambda s: word_tok(s.lower())
        detok   = lambda toks: " ".join(toks)
        return src_tok, tgt_tok, detok, src_vec, tgt_vec
    if exp == "bpemb":
        bp_src, bp_tgt = load_bpemb(ISO_SRC), load_bpemb(ISO_TGT)
        print(bp_src, "|", bp_tgt)
        src_vec = write_w2v(os.path.join(EMB_DIR, f"bpemb.{ISO_SRC}.vs{BPEMB_VS}.d{EMB_DIM}.txt"), list(bp_src.pieces), bp_src.vectors)
        tgt_vec = write_w2v(os.path.join(EMB_DIR, f"bpemb.{ISO_TGT}.vs{BPEMB_VS}.d{EMB_DIM}.txt"), list(bp_tgt.pieces), bp_tgt.vectors)
        src_tok = bp_src.encode          # text -> subword pieces (BPEmb lower-cases and maps digits to 0)
        tgt_tok = bp_tgt.encode
        detok   = bp_tgt.spm.DecodePieces
        return src_tok, tgt_tok, detok, src_vec, tgt_vec
    raise ValueError(exp)

# quick look at the two kinds of tokens
demo = "Hearing loss is common in older people."
print("fasttext tokens:", word_tok(demo.lower()))

fasttext tokens: ['hearing', 'loss', 'is', 'common', 'in', 'older', 'people', '.']


## 7. Config, vocabulary, training, translation and evaluation
`run_experiment(exp)` runs the whole pipeline for one embedding type:
1. tokenise and write the data files
2. build the OpenNMT-py vocabulary
3. train a Transformer whose encoder/decoder embeddings are **initialised from the pretrained vectors** (`src_embeddings`, `tgt_embeddings`, `embeddings_type: word2vec`; tokens missing from the embedding file start as zero vectors)
4. translate the test set (beam search)
5. compute BLEU / chrF overall and per bin

In [8]:
def run_onmt(module, *args):
    """Run an OpenNMT-py command with the OpenNMT-py Python (ONMT_PY) and stream its output."""
    cmd = [ONMT_PY, "-m", f"onmt.bin.{module}", *args]
    print(" ".join(cmd))
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        print(line, end="")
        tail.append(line); tail = tail[-30:]
    p.wait()
    if p.returncode != 0:
        raise RuntimeError(f"onmt.bin.{module} failed (exit {p.returncode}). Last output:\n" + "".join(tail))

def read_vocab_tokens(path, cap):
    toks = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                toks.append(line.split()[0])
    return toks[:cap]

def write_configs(wd, src_emb, tgt_emb):
    gpu = "gpu_ranks: [0]" if USE_GPU else "gpu_ranks: []"
    freeze = str(FREEZE_EMB).lower()
    data_block = f"""
data:
    corpus_1:
        path_src: {wd}/train.src
        path_tgt: {wd}/train.tgt
        weight: 1
    valid:
        path_src: {wd}/valid.src
        path_tgt: {wd}/valid.tgt
"""
    vocab_cfg = f"""
save_data: {wd}/onmt_run
src_vocab: {wd}/vocab.src
tgt_vocab: {wd}/vocab.tgt
overwrite: true
skip_empty_level: silent
""" + data_block
    train_cfg = f"""
save_data: {wd}/onmt_run
src_vocab: {wd}/vocab.src
tgt_vocab: {wd}/vocab.tgt
src_vocab_size: {WORD_VOCAB_SIZE}
tgt_vocab_size: {WORD_VOCAB_SIZE}
overwrite: true
skip_empty_level: silent
seed: {SEED}
""" + data_block + f"""
# pretrained embeddings (word2vec text format) for encoder and decoder
src_embeddings: {src_emb}
tgt_embeddings: {tgt_emb}
embeddings_type: "word2vec"
freeze_word_vecs_enc: {freeze}
freeze_word_vecs_dec: {freeze}

save_model: {wd}/model
save_checkpoint_steps: 1000
keep_checkpoint: 2
train_steps: {TRAIN_STEPS}
valid_steps: 1000
report_every: 100

world_size: 1
{gpu}

batch_type: tokens
batch_size: 2048
valid_batch_size: 16
accum_count: [1]
accum_steps: [0]
queue_size: 1000
bucket_size: 10000
num_workers: 0

optim: adam
adam_beta1: 0.9
adam_beta2: 0.998
decay_method: noam
learning_rate: 2.0
warmup_steps: 1000
max_grad_norm: 0.0
label_smoothing: 0.1
param_init: 0.0
param_init_glorot: true
normalization: tokens

encoder_type: transformer
decoder_type: transformer
position_encoding: true
enc_layers: {LAYERS}
dec_layers: {LAYERS}
heads: {HEADS}
hidden_size: {EMB_DIM}
word_vec_size: {EMB_DIM}
transformer_ff: {FF_SIZE}
dropout_steps: [0]
dropout: [0.1]
attention_dropout: [0.1]
share_vocab: false
"""
    with open(f"{wd}/config.vocab.yaml", "w", encoding="utf-8") as f: f.write(vocab_cfg)
    with open(f"{wd}/config.train.yaml", "w", encoding="utf-8") as f: f.write(train_cfg)

def step_of(p):
    m = re.search(r"model_step_(\d+)\.pt$", p)
    return int(m.group(1)) if m else -1

BLEU_METRIC, CHRF_METRIC = BLEU(tokenize="none"), CHRF()

def run_experiment(exp):
    t0 = time.time()
    wd = os.path.join(WORKDIR, exp)
    os.makedirs(wd, exist_ok=True)
    print(f"\n{'=' * 70}\nEXPERIMENT: {exp}\n{'=' * 70}")
    src_tok, tgt_tok, detok, src_emb, tgt_emb = get_setup(exp)

    # ---- 1. tokenise + write data files ----
    def tok_lines(tok, texts):
        return [" ".join(tok(s)) or "<unk>" for s in texts]
    for name, d in [("train", train_df), ("valid", valid_df)]:
        write_lines(f"{wd}/{name}.src", tok_lines(src_tok, d["src"]))
        write_lines(f"{wd}/{name}.tgt", tok_lines(tgt_tok, d["tgt"]))
    test_src_lines = tok_lines(src_tok, test_df["src"])
    write_lines(f"{wd}/test.src", test_src_lines)
    print("Example source tokens:", test_src_lines[0][:150])

    # ---- 2. vocabulary ----
    write_configs(wd, src_emb, tgt_emb)
    run_onmt("build_vocab", "-config", f"{wd}/config.vocab.yaml", "-n_sample", "-1")
    src_vocab = read_vocab_tokens(f"{wd}/vocab.src", WORD_VOCAB_SIZE - 4)
    tgt_vocab = read_vocab_tokens(f"{wd}/vocab.tgt", WORD_VOCAB_SIZE - 4)
    emb_src_tokens, emb_tgt_tokens = read_vec_tokens(src_emb), read_vec_tokens(tgt_emb)
    cov_src = np.mean([t in emb_src_tokens for t in src_vocab])
    cov_tgt = np.mean([t in emb_tgt_tokens for t in tgt_vocab])
    print(f"Vocab sizes: src={len(src_vocab)} tgt={len(tgt_vocab)} | pretrained coverage: src={cov_src:.1%} tgt={cov_tgt:.1%}")

    # ---- 3. training ----
    final_ckpt = f"{wd}/model_step_{TRAIN_STEPS}.pt"
    if os.path.exists(final_ckpt) and not FORCE_RERUN:
        print("Final checkpoint exists, skipping training:", final_ckpt)
    else:
        run_onmt("train", "-config", f"{wd}/config.train.yaml")
    t_train = time.time() - t0

    # ---- 4. translation ----
    ckpts = sorted(glob.glob(f"{wd}/model_step_*.pt"), key=step_of)
    assert ckpts, "No checkpoint found; training did not finish."
    best_ckpt = ckpts[-1]
    print("Using checkpoint:", best_ckpt)
    hyp_path = f"{wd}/test.tok.hyp"
    done = os.path.exists(hyp_path) and sum(1 for _ in open(hyp_path, encoding="utf-8")) == len(test_df)
    if done and not FORCE_RERUN:
        print("Translations exist, skipping:", hyp_path)
    else:
        args = ["-model", best_ckpt, "-src", f"{wd}/test.src", "-output", hyp_path,
                "-beam_size", str(BEAM_SIZE), "-batch_size", "32", "-batch_type", "sents",
                "-max_length", "200"]
        if USE_GPU: args += ["-gpu", "0"]
        run_onmt("translate", *args)
    with open(hyp_path, encoding="utf-8") as f:
        hyp_toks = [l.rstrip("\n").split() for l in f]
    assert len(hyp_toks) == len(test_df), (len(hyp_toks), len(test_df))
    hyps = [detok([t for t in toks if t != "<unk>"]) for toks in hyp_toks]

    # ---- 5. evaluation (same normalisation for refs and hyps) ----
    res_df = test_df.copy()
    res_df["hyp"] = hyps
    res_df["ref_n"] = res_df["tgt"].apply(norm_eval)
    res_df["hyp_n"] = res_df["hyp"].apply(norm_eval)
    src_vocab_set = set(src_vocab)
    res_df["src_oov"] = [np.mean([t not in src_vocab_set for t in l.split()]) if l.split() else 0.0 for l in test_src_lines]

    overall_bleu = BLEU_METRIC.corpus_score(res_df["hyp_n"].tolist(), [res_df["ref_n"].tolist()])
    overall_chrf = CHRF_METRIC.corpus_score(res_df["hyp_n"].tolist(), [res_df["ref_n"].tolist()])
    rows = []
    for b in ["bin1", "bin2", "bin3", "bin4"]:
        g = res_df[res_df["bin"] == b]
        s = BLEU_METRIC.corpus_score(g["hyp_n"].tolist(), [g["ref_n"].tolist()])
        c = CHRF_METRIC.corpus_score(g["hyp_n"].tolist(), [g["ref_n"].tolist()])
        rows.append({"bin": b, "test sentences": len(g), "avg src length": round(g["len"].mean(), 1),
                     "BLEU": round(s.score, 2), "chrF": round(c.score, 2),
                     "src OOV %": round(100 * g["src_oov"].mean(), 2)})
    bin_results = pd.DataFrame(rows)

    res_df[["src", "tgt", "hyp", "bin"]].to_csv(f"{wd}/test_translations.csv", index=False)
    bin_results.to_csv(f"{wd}/bin_results.csv", index=False)
    print(f"\n[{exp}] Overall BLEU: {overall_bleu}")
    print(f"[{exp}] Overall chrF: {overall_chrf}")
    print(bin_results.to_string(index=False))
    return {"exp": exp, "overall_bleu": overall_bleu.score, "overall_chrf": overall_chrf.score,
            "bin_results": bin_results, "translations": res_df, "minutes": (time.time() - t0) / 60,
            "vocab": (len(src_vocab), len(tgt_vocab)), "coverage": (cov_src, cov_tgt)}

RESULTS = {}

## 8. Experiment 1: FastText embeddings (word-level)

In [9]:
RESULTS["fasttext"] = run_experiment("fasttext")2


EXPERIMENT: fasttext
Streaming https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.en.vec (stops after 200000 vectors)
  kept 50000
  kept 100000
  kept 150000
  kept 200000
Saved 200000 vectors of dim 300 -> mt_emb_work/embeddings/ft.en.vec
Streaming https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.hi.vec (stops after 200000 vectors)
  kept 50000
  kept 100000
  kept 150000
Saved 158016 vectors of dim 300 -> mt_emb_work/embeddings/ft.hi.vec
Example source tokens: which has clinical fruit smell and sharp taste .
/kaggle/working/onmt_py311/bin/python -m onmt.bin.build_vocab -config mt_emb_work/fasttext/config.vocab.yaml -n_sample -1
[2026-11-03 14:54:40,198 INFO] Counter vocab from -1 samples.
[2026-11-03 14:54:40,199 INFO] n_sample=-1: Build vocab on full datasets.
[2026-11-03 14:54:40,953 INFO] Counters src: 28497
[2026-11-03 14:54:40,954 INFO] Counters tgt: 36711
Vocab sizes: src=28497 tgt=29996 | pretrained coverage: src=76.6% tgt=68.5%
/kaggle/working/onmt_py311/bin

## 9. Experiment 2: BPEmb subword embeddings

In [10]:
RESULTS["bpemb"] = run_experiment("bpemb")


EXPERIMENT: bpemb
downloading https://nlp.h-its.org/bpemb/en/en.wiki.bpe.vs10000.model


100%|██████████| 400869/400869 [00:00<00:00, 705122.39B/s]


downloading https://nlp.h-its.org/bpemb/en/en.wiki.bpe.vs10000.d300.w2v.bin.tar.gz


100%|██████████| 11189884/11189884 [00:03<00:00, 3682071.54B/s]


downloading https://nlp.h-its.org/bpemb/hi/hi.wiki.bpe.vs10000.model


100%|██████████| 463471/463471 [00:00<00:00, 642571.12B/s]


downloading https://nlp.h-its.org/bpemb/hi/hi.wiki.bpe.vs10000.d300.w2v.bin.tar.gz


100%|██████████| 11215592/11215592 [00:01<00:00, 8383883.67B/s]


BPEmb(lang=en, vs=10000, dim=300) | BPEmb(lang=hi, vs=10000, dim=300)
Saved 9997 vectors -> mt_emb_work/embeddings/bpemb.en.vs10000.d300.txt
Saved 9997 vectors -> mt_emb_work/embeddings/bpemb.hi.vs10000.d300.txt
Example source tokens: ▁which ▁has ▁clinical ▁fruit ▁sm ell ▁and ▁sharp ▁t aste .
/kaggle/working/onmt_py311/bin/python -m onmt.bin.build_vocab -config mt_emb_work/bpemb/config.vocab.yaml -n_sample -1
[2026-11-03 15:08:49,534 INFO] Counter vocab from -1 samples.
[2026-11-03 15:08:49,534 INFO] n_sample=-1: Build vocab on full datasets.
[2026-11-03 15:08:50,391 INFO] Counters src: 8918
[2026-11-03 15:08:50,391 INFO] Counters tgt: 8996
Vocab sizes: src=8918 tgt=8996 | pretrained coverage: src=99.6% tgt=99.4%
/kaggle/working/onmt_py311/bin/python -m onmt.bin.train -config mt_emb_work/bpemb/config.train.yaml
[2026-11-03 15:08:53,253 INFO] Missing transforms field for corpus_1 data, set to default: [].
[2026-11-03 15:08:53,253 INFO] Missing transforms field for valid data, set to def

## 10. Bin-wise comparison of the two embeddings

In [11]:
summary = pd.DataFrame({
    "bin": ["bin1", "bin2", "bin3", "bin4", "overall"],
})
for exp, res in RESULTS.items():
    br = res["bin_results"].set_index("bin")
    summary[f"{exp} BLEU"] = [br.loc[b, "BLEU"] for b in ["bin1", "bin2", "bin3", "bin4"]] + [round(res["overall_bleu"], 2)]
    summary[f"{exp} chrF"] = [br.loc[b, "chrF"] for b in ["bin1", "bin2", "bin3", "bin4"]] + [round(res["overall_chrf"], 2)]
summary.insert(1, "test sentences", [500, 500, 500, 500, 2000] if MAX_TEST_PER_BIN == 500 else [None] * 5)
summary

,bin,test sentences,fasttext BLEU,fasttext chrF,bpemb BLEU,bpemb chrF
0,bin1,500,13.80,26.52,12.13,25.79
1,bin2,500,9.93,25.04,9.27,23.39
2,bin3,500,7.55,23.99,7.20,22.41
3,bin4,500,8.83,25.20,6.59,21.61
4,overall,2000,9.11,24.94,7.75,22.56


In [12]:
for exp, res in RESULTS.items():
    br = res["bin_results"]
    best, worst = br.loc[br["BLEU"].idxmax()], br.loc[br["BLEU"].idxmin()]
    trend = br["BLEU"].diff().dropna()
    print(f"[{exp}] overall BLEU {res['overall_bleu']:.2f} | highest: {best['bin']} ({best['BLEU']}), lowest: {worst['bin']} ({worst['BLEU']})")
    print(f"[{exp}] BLEU by bin (short -> long): {br['BLEU'].tolist()} | chrF by bin: {br['chrF'].tolist()}")
    if (trend < 0).all():   print(f"[{exp}] BLEU decreases steadily as sentence length increases.")
    elif (trend > 0).all(): print(f"[{exp}] BLEU increases steadily as sentence length increases.")
    else:                   print(f"[{exp}] BLEU does not change monotonically with sentence length.")
    print(f"[{exp}] vocab (src, tgt): {res['vocab']} | pretrained coverage (src, tgt): {res['coverage'][0]:.1%}, {res['coverage'][1]:.1%} | total time: {res['minutes']:.0f} min")
    print(f"[{exp}] source OOV % per bin: {br['src OOV %'].tolist()}\n")

if len(RESULTS) == 2:
    a, b = list(RESULTS)
    winner = a if RESULTS[a]["overall_bleu"] >= RESULTS[b]["overall_bleu"] else b
    print(f"Higher overall BLEU: {winner} ({RESULTS[winner]['overall_bleu']:.2f})")

[fasttext] overall BLEU 9.11 | highest: bin1 (13.8), lowest: bin3 (7.55)
[fasttext] BLEU by bin (short -> long): [13.8, 9.93, 7.55, 8.83] | chrF by bin: [26.52, 25.04, 23.99, 25.2]
[fasttext] BLEU does not change monotonically with sentence length.
[fasttext] vocab (src, tgt): (28497, 29996) | pretrained coverage (src, tgt): 76.6%, 68.5% | total time: 14 min
[fasttext] source OOV % per bin: [3.77, 3.18, 3.08, 2.6]

[bpemb] overall BLEU 7.75 | highest: bin1 (12.13), lowest: bin4 (6.59)
[bpemb] BLEU by bin (short -> long): [12.13, 9.27, 7.2, 6.59] | chrF by bin: [25.79, 23.39, 22.41, 21.61]
[bpemb] BLEU decreases steadily as sentence length increases.
[bpemb] vocab (src, tgt): (8918, 8996) | pretrained coverage (src, tgt): 99.6%, 99.4% | total time: 7 min
[bpemb] source OOV % per bin: [0.18, 0.14, 0.14, 0.1]

Higher overall BLEU: fasttext (9.11)


In [13]:
# a few example translations from each system
for exp, res in RESULTS.items():
    print(f"\n===== {exp} =====")
    for _, r in res["translations"].groupby("bin").head(1).iterrows():
        print(f"[{r['bin']}] SRC: {r['src']}\n       REF: {r['tgt']}\n       HYP: {r['hyp']}\n")


===== fasttext =====
[bin1] SRC: Which has clinical fruit smell and sharp taste.
       REF: जिसमें लाक्षणिक फल गंध और तीखा स्वाद होता है।
       HYP: जो फल और क्लिनिकल विशेषताएँ हैं ।

[bin2] SRC: On the Ramakrishna Beach of Vishakhapattanam this wish of yours can get fulfilled .
       REF: क्‍या आपने कभी पनडुब्बी के अंदर की दुनिया देखी है ?
       HYP: विशाखापट्नम के रामकृष्‍ण बीच पर आपकी यह इच्छा पूरी हो सकती है ।

[bin3] SRC: The forest animal found here are Andamanese pig , dugong , dolphin , water monitor , several species of snake and reptiles , turtle , crocodile etc .
       REF: यहाँ पाए जाने वाले वन्य पशु अंडमानी सुअर , डुगांग , डॉल्फिन , जल गोह ( वाटर मोनीटर ) कई जातियों के सर्प व सरीसृप कछुवे , घाड़ियाल आदि हैं ।
       HYP: सैडलपीक राष्‍ट्रीय उद्यान में डुगोंग , डॉल्फिन , मगरमच्छ , घड़ियाल , जलगोह , आलिव रिडले कछुवा बहुतायात में पाए जाते हैं ।

[bin4] SRC: 9 Days / 8 Nights US$ 499 The itinerary along the banks of holy rivers brings you the yoga experience at Varanasi Gha

## 11. Bin-wise result analysis
**Analysis (fill in after running):** Compare BLEU (and chrF) across bin1–bin4 for each embedding type, then compare the two embeddings. Points to discuss:
* how BLEU changes as the source length grows (long sentences usually score lower because errors accumulate and the model sees fewer long examples);
* whether very short sentences score unusually high or low (few n-grams make BLEU unstable);
* the number of test sentences per bin (small bins give noisy scores) and the amount of training data per bin;
* **FastText (word-level)** vs **BPEmb (subword)**: look at the *src OOV %* column and the vocabulary coverage. Word-level models have large vocabularies and out-of-vocabulary words (removed as `<unk>` in the output), while subword models have almost no OOV but longer sequences;
* whether pretrained embeddings help with such a small model and training set (this CPU run is small, so absolute BLEU will be low).

In [16]:
import os, glob, shutil, zipfile
from IPython.display import FileLink, display

OUT_DIR = "submission_outputs"
ZIP_NAME = "illm_assignment_outputs.zip"

shutil.rmtree(OUT_DIR, ignore_errors=True)
os.makedirs(OUT_DIR, exist_ok=True)

# 1. Summary tables and translations
summary.to_csv(f"{OUT_DIR}/summary_binwise_comparison.csv", index=False)
for exp, res in RESULTS.items():
    res["bin_results"].to_csv(f"{OUT_DIR}/{exp}_bin_results.csv", index=False)
    res["translations"][["src", "tgt", "hyp", "bin"]].to_csv(
        f"{OUT_DIR}/{exp}_test_translations.csv", index=False, encoding="utf-8")

# 2. Plain-text report
with open(f"{OUT_DIR}/results_report.txt", "w", encoding="utf-8") as f:
    f.write("English -> Hindi NMT: FastText vs BPEmb\n" + "=" * 50 + "\n\n")
    f.write(summary.to_string(index=False) + "\n\n")
    for exp, res in RESULTS.items():
        f.write(f"[{exp}] overall BLEU={res['overall_bleu']:.2f} chrF={res['overall_chrf']:.2f} "
                f"| vocab(src,tgt)={res['vocab']} "
                f"| pretrained coverage={res['coverage'][0]:.1%}/{res['coverage'][1]:.1%} "
                f"| time={res['minutes']:.1f} min\n")
        f.write(res["bin_results"].to_string(index=False) + "\n\n")

# 3. Small per-experiment files only (no checkpoints)
for exp in RESULTS:
    src, dst = os.path.join(WORKDIR, exp), os.path.join(OUT_DIR, exp)
    os.makedirs(dst, exist_ok=True)
    for pat in ["config.*.yaml", "train.src", "train.tgt", "valid.src", "valid.tgt",
                "test.src", "test.tok.hyp", "vocab.src", "vocab.tgt"]:
        for p in glob.glob(os.path.join(src, pat)):
            shutil.copy2(p, dst)

# 4. Zip
if os.path.exists(ZIP_NAME):
    os.remove(ZIP_NAME)
with zipfile.ZipFile(ZIP_NAME, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT_DIR):
        for fn in files:
            full = os.path.join(root, fn)
            z.write(full, os.path.relpath(full, OUT_DIR))

print(f"Created {ZIP_NAME} ({os.path.getsize(ZIP_NAME)/1e6:.1f} MB)")
display(FileLink(ZIP_NAME))

Created illm_assignment_outputs.zip (6.8 MB)


/kaggle/working/illm_assignment_outputs.zip